In [71]:
import os
import pickle
import langchain
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

In [72]:
# This replaces the chain's internal state
class RAGState(TypedDict):
    question: str
    documents: List[str]
    sources: List[str]
    answer: str

In [73]:
# Load URLs
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(data)

# Embed & store
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# Save using FAISS built-in method
vectorstore.save_local("faiss_index")

In [74]:
llm = OllamaLLM(model="llama3", temperature=0.9)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context.
Also mention the sources used at the end.

Context:
{context}

Question: {question}

Answer:
""")

In [75]:
# Node 1: Retrieve relevant documents
def retrieve(state: RAGState) -> RAGState:
    question = state["question"]
    retrieved_docs = retriever.invoke(question)
    
    documents = [doc.page_content for doc in retrieved_docs]
    sources = [doc.metadata.get("source", "unknown") for doc in retrieved_docs]
    
    return {**state, "documents": documents, "sources": sources}


# Node 2: Generate answer using LLM
def generate(state: RAGState) -> RAGState:
    context = "\n\n".join(state["documents"])
    sources = list(set(state["sources"]))  # deduplicate
    
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": state["question"]})
    
    # Append sources to answer (mimics RetrievalQAWithSourcesChain output)
    answer_with_sources = f"{answer}\n\nSources:\n" + "\n".join(f"- {s}" for s in sources)
    
    return {**state, "answer": answer_with_sources}

In [76]:
graph = StateGraph(RAGState)

# Add nodes
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

# Define flow: retrieve → generate → END
graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

# Compile
rag_app = graph.compile()

In [77]:
# Load from disk (optional, if restarting)
if os.path.exists("faiss_index"):
    vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
    retriever = vectorstore.as_retriever()

query = "what is the price of Tiago iCNG?"

result = rag_app.invoke({"question": query, "documents": [], "sources": [], "answer": ""})

print(result["answer"])

The price of Tiago iCNG ranges from Rs 6.55 lakh to Rs 8.1 lakh.

Sources:
- https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html
- https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html
